<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/LayoutLMv3/Fine_tune_LayoutLMv3_on_FUNSD_(HuggingFace_Trainer).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Set-up environment

First, we install ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â°ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¸ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¤ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â Transformers, as well as ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â°ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¸ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¤ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â Datasets and Seqeval (the latter is useful for evaluation metrics such as F1 on sequence labeling tasks).

In [ ]:
# CHANGED CELL: dependency check (no auto-reinstall by default)
import sys
import subprocess
import importlib

AUTO_INSTALL_MISSING = False  # set True only once if you want this cell to install missing deps

REQUIRED_PACKAGES = [
    ("datasets", "datasets>=2.16.0"),
    ("evaluate", "evaluate>=0.4.0"),
    ("seqeval", "seqeval>=1.2.2"),
    ("transformers", "transformers>=4.40.0"),
    ("accelerate", "accelerate>=0.28.0"),
    ("PIL", "pillow>=10.0.0"),
]

missing = []
for import_name, requirement in REQUIRED_PACKAGES:
    try:
        importlib.import_module(import_name)
        print(f"OK: {requirement}")
    except Exception:
        print(f"MISSING: {requirement}")
        missing.append(requirement)

if missing:
    if AUTO_INSTALL_MISSING:
        print("Installing missing packages...")
        for requirement in missing:
            subprocess.check_call([sys.executable, "-m", "pip", "install", requirement])
        print("Done. Restart kernel after installs.")
    else:
        print("\nMissing dependencies detected.")
        print("Install once from terminal with:")
        print(f"{sys.executable} -m pip install " + " ".join(missing))
else:
    print("All required notebook deps are installed.")

In [ ]:
# ADDED CELL: quick package versions
import transformers, datasets, evaluate, seqeval, PIL
from importlib.metadata import version as pkg_version, PackageNotFoundError


def safe_pkg_version(name, fallback_attr_module=None):
    try:
        return pkg_version(name)
    except PackageNotFoundError:
        if fallback_attr_module is not None:
            return getattr(fallback_attr_module, "__version__", "unknown")
        return "unknown"


print("transformers:", safe_pkg_version("transformers", transformers))
print("datasets:", safe_pkg_version("datasets", datasets))
print("evaluate:", safe_pkg_version("evaluate", evaluate))
print("seqeval:", safe_pkg_version("seqeval", seqeval))
print("pillow:", safe_pkg_version("pillow", PIL))

## Load dataset

Next, we load the local dataset from your machine. The notebook now defaults to the folder at C:\Users\raefh\Desktop\gen\dataset, and it can also fall back to the archive at C:\Users\raefh\Desktop\gen\dataset.rar if needed.

In [ ]:
# CHANGED CELL: imports for local CNOPT dataset + NER + doc-status classification
import os
import glob
import json
import re
import random
import sys
import subprocess
import inspect
from collections import Counter

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

from datasets import Dataset, DatasetDict, Features, Sequence, Value, Image as HFImage
import evaluate

from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    default_data_collator,
    EarlyStoppingCallback,
    set_seed,
)

In [ ]:
# CHANGED CELL: dataset root setup (your local dataset + optional archive extraction)
RAW_DATASET_ROOT = os.getenv(
    "CNOPT_DATASET_ROOT",
    r"C:\Users\raefh\Desktop\gen\dataset" if os.name == "nt" else "/content/dataset",
)
RAW_ARCHIVE_PATH = os.getenv(
    "CNOPT_DATASET_ARCHIVE",
    os.getenv(
        "CNOPT_DATASET_ZIP",
        r"C:\Users\raefh\Desktop\gen\dataset.rar" if os.name == "nt" else "/content/dataset.zip",
    ),
)

EXPECTED_SPLITS = ("train", "val", "test")


def _split_path(root_dir, folder_name, split_name):
    return os.path.join(root_dir, folder_name, split_name)


def _has_dataset_layout(root_dir):
    if not os.path.isdir(root_dir):
        return False
    return all(
        os.path.isdir(_split_path(root_dir, "annotations", split))
        and os.path.isdir(_split_path(root_dir, "images", split))
        for split in EXPECTED_SPLITS
    )


def _resolve_dataset_root(root_dir):
    if _has_dataset_layout(root_dir):
        return root_dir
    if not os.path.isdir(root_dir):
        return root_dir

    for name in sorted(os.listdir(root_dir)):
        child = os.path.join(root_dir, name)
        if _has_dataset_layout(child):
            return child

    return root_dir


def _extract_dataset_archive(archive_path, target_dir):
    ext = os.path.splitext(archive_path)[1].lower()

    if ext == ".zip":
        import zipfile

        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(target_dir)
        return

    if ext == ".rar":
        try:
            import rarfile
        except ModuleNotFoundError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "rarfile"])
            import rarfile

        try:
            with rarfile.RarFile(archive_path, "r") as rf:
                rf.extractall(target_dir)
        except Exception as exc:
            raise RuntimeError(
                "RAR extraction failed. Install an extraction backend (unrar/bsdtar/unar), "
                "or extract manually and point CNOPT_DATASET_ROOT to the extracted folder."
            ) from exc
        return

    raise ValueError(f"Unsupported archive extension: {ext}. Use .zip or .rar.")


RAW_DATASET_ROOT = _resolve_dataset_root(RAW_DATASET_ROOT)
if not _has_dataset_layout(RAW_DATASET_ROOT):
    if not os.path.isfile(RAW_ARCHIVE_PATH):
        raise FileNotFoundError(
            f"Dataset folder not found/invalid at {RAW_DATASET_ROOT}, and archive not found at {RAW_ARCHIVE_PATH}. "
            "Set CNOPT_DATASET_ROOT or CNOPT_DATASET_ARCHIVE/CNOPT_DATASET_ZIP."
        )

    os.makedirs(RAW_DATASET_ROOT, exist_ok=True)
    _extract_dataset_archive(RAW_ARCHIVE_PATH, RAW_DATASET_ROOT)
    RAW_DATASET_ROOT = _resolve_dataset_root(RAW_DATASET_ROOT)

if not _has_dataset_layout(RAW_DATASET_ROOT):
    raise FileNotFoundError(
        f"Dataset structure invalid at {RAW_DATASET_ROOT}. "
        "Expected annotations/{train,val,test} and images/{train,val,test}."
    )

DATASET_ROOT = RAW_DATASET_ROOT
print("DATASET_ROOT:", DATASET_ROOT)

for split_name in EXPECTED_SPLITS:
    ann_count = len(glob.glob(os.path.join(DATASET_ROOT, "annotations", split_name, "*.json")))
    img_count = len(glob.glob(os.path.join(DATASET_ROOT, "images", split_name, "*.*")))
    print(f"{split_name}: annotations={ann_count}, images={img_count}")

In [ ]:
# CHANGED CELL: helpers + label/status space discovery from your dataset
PREFERRED_DOC_STATUS_ORDER = ["valid", "suspicious", "incomplete", "wrong_type"]


def _read_json(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return json.load(f)


def _clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def _normalize_bbox(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        box = [0, 0, 0, 0]

    x0, y0, x1, y1 = [float(v) for v in box]
    x0 = max(0.0, min(x0, float(width)))
    x1 = max(0.0, min(x1, float(width)))
    y0 = max(0.0, min(y0, float(height)))
    y1 = max(0.0, min(y1, float(height)))

    if width <= 0 or height <= 0:
        return [0, 0, 0, 0]

    return [
        int(1000 * x0 / width),
        int(1000 * y0 / height),
        int(1000 * x1 / width),
        int(1000 * y1 / height),
    ]


def _parse_tokens(ann):
    raw_tokens = ann.get("tokens", [])

    if raw_tokens and isinstance(raw_tokens[0], dict):
        for token in raw_tokens:
            yield (
                _clean_text(token.get("text", "")),
                token.get("bbox", token.get("bboxes", [0, 0, 0, 0])),
                _clean_text(token.get("label", token.get("labels", "O"))) or "O",
            )
        return

    if raw_tokens and isinstance(raw_tokens[0], str):
        raw_boxes = ann.get("bboxes", ann.get("bbox", [[0, 0, 0, 0] for _ in raw_tokens]))
        raw_labels = ann.get("labels", ["O" for _ in raw_tokens])
        for text, bbox, label in zip(raw_tokens, raw_boxes, raw_labels):
            yield (_clean_text(text), bbox, _clean_text(label) or "O")


def _discover_label_and_status_space(root_dir):
    label_set = set()
    status_set = set()

    for split_name in EXPECTED_SPLITS:
        ann_dir = os.path.join(root_dir, "annotations", split_name)
        for ann_path in sorted(glob.glob(os.path.join(ann_dir, "*.json"))):
            ann = _read_json(ann_path)

            doc_status = _clean_text(ann.get("doc_status", "valid")).lower() or "valid"
            status_set.add(doc_status)

            for _, _, label in _parse_tokens(ann):
                label_set.add(label or "O")

    if "O" not in label_set:
        label_set.add("O")

    ordered_labels = ["O"] + sorted([lbl for lbl in label_set if lbl != "O"])
    ordered_status = [s for s in PREFERRED_DOC_STATUS_ORDER if s in status_set]
    ordered_status.extend(sorted([s for s in status_set if s not in ordered_status]))

    return ordered_labels, ordered_status


label_list, doc_status_list = _discover_label_and_status_space(DATASET_ROOT)

label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}
num_labels = len(label_list)

status2id = {status: idx for idx, status in enumerate(doc_status_list)}
id2status = {idx: status for status, idx in status2id.items()}
doc_label2id = status2id
ndoc_labels = len(doc_status_list)
doc_id2label = id2status

print("Token labels:", label_list)
print("Document statuses:", doc_status_list)

In [ ]:
# CHANGED CELL: load dataset in HF format (NER labels + doc_status)

def load_split(split_name):
    ann_dir = os.path.join(DATASET_ROOT, "annotations", split_name)
    rows = []
    skipped_missing_image = 0
    skipped_empty_tokens = 0

    for ann_path in sorted(glob.glob(os.path.join(ann_dir, "*.json"))):
        ann = _read_json(ann_path)

        image_rel = _clean_text(ann.get("image", "")).replace("\\", "/")
        image_path = os.path.join(DATASET_ROOT, image_rel)
        if not os.path.isfile(image_path):
            image_basename = os.path.basename(image_rel)
            fallback = os.path.join(DATASET_ROOT, "images", split_name, image_basename)
            if os.path.isfile(fallback):
                image_path = fallback
            else:
                skipped_missing_image += 1
                continue

        with Image.open(image_path) as img:
            width, height = img.size

        tokens = []
        bboxes = []
        ner_tags_str = []

        for text, raw_bbox, raw_label in _parse_tokens(ann):
            text = _clean_text(text)
            if not text:
                continue
            tokens.append(text)
            bboxes.append(_normalize_bbox(raw_bbox, width, height))
            ner_tags_str.append(raw_label if raw_label in label2id else "O")

        if not tokens:
            skipped_empty_tokens += 1
            continue

        doc_status = _clean_text(ann.get("doc_status", "valid")).lower() or "valid"
        if doc_status not in status2id:
            doc_status = doc_status_list[0]

        rows.append(
            {
                "id": _clean_text(ann.get("id", os.path.splitext(os.path.basename(ann_path))[0])),
                "image": image_path,
                "doc_status": doc_status,
                "tokens": tokens,
                "bboxes": bboxes,
                "ner_tags_str": ner_tags_str,
            }
        )

    print(
        f"{split_name}: loaded={len(rows)}, "
        f"skipped_missing_image={skipped_missing_image}, "
        f"skipped_empty_tokens={skipped_empty_tokens}"
    )
    return rows


features = Features(
    {
        "id": Value("string"),
        "image": HFImage(),
        "doc_status": Value("string"),
        "tokens": Sequence(Value("string")),
        "bboxes": Sequence(Sequence(Value("int64"), length=4)),
        "ner_tags_str": Sequence(Value("string")),
    }
)


dataset = DatasetDict(
    {
        "train": Dataset.from_list(load_split("train"), features=features),
        "validation": Dataset.from_list(load_split("val"), features=features),
        "test": Dataset.from_list(load_split("test"), features=features),
    }
)


def _map_training_labels(example):
    example["ner_tags"] = [label2id.get(lbl, label2id["O"]) for lbl in example["ner_tags_str"]]
    status = example["doc_status"]
    if status not in status2id:
        status = doc_status_list[0]
    example["doc_status"] = status
    example["doc_status_id"] = status2id[status]
    return example


dataset = dataset.map(_map_training_labels)
print(dataset)

In [ ]:
# CHANGED CELL: small debug view (sample + tokens + boxes + label ids + image size + doc_status)
sample = dataset["train"][0]
print("id:", sample["id"])
print("doc_status:", sample.get("doc_status"))
print("tokens[:12]:", sample["tokens"][:12])
print("bboxes[:12]:", sample["bboxes"][:12])
print("label_ids[:12]:", sample["ner_tags"][:12])
print("image size:", sample["image"].size)

As we can see, the dataset consists of 2 splits ("train" and "test"), and each example contains a list of words ("tokens") with corresponding boxes ("bboxes"), and the words are tagged ("ner_tags"). Each example also include the original image ("image").

In [ ]:
# CHANGED CELL: compact dataset diagnostics (sizes + status balance + token labels)
print(dataset)

for split_name in ["train", "validation", "test"]:
    split_ds = dataset[split_name]
    status_counts = Counter(split_ds["doc_status"])

    token_label_counts = Counter()
    for ex in split_ds:
        token_label_counts.update(ex["ner_tags_str"])

    print(f"\n[{split_name}] docs={len(split_ds)}")
    print("doc_status counts:", dict(status_counts))
    print("token label counts:", dict(sorted(token_label_counts.items())))

Let's check the features:

In [ ]:
dataset["train"].features

Note that you can directly see the example in a notebook (as the "image" column is of type [Image](https://huggingface.co/docs/datasets/v2.2.1/en/package_reference/main_classes#datasets.Image)).

In [ ]:
example = dataset["train"][0]
example["image"]

In [ ]:
words, boxes, ner_tags = example["tokens"], example["bboxes"], example["ner_tags"]
print(words)
print(boxes)
print(ner_tags)

## Prepare dataset

Next, we prepare the dataset for the model. This can be done very easily using `LayoutLMv3Processor`, which internally wraps a `LayoutLMv3FeatureExtractor` (for the image modality) and a `LayoutLMv3Tokenizer` (for the text modality) into one.

Basically, the processor does the following internally:
* the feature extractor is used to resize + normalize each document image into `pixel_values`
* the tokenizer is used to turn the words, boxes and NER tags into token-level `input_ids`, `attention_mask` and `labels`.

The processor simply returns a dictionary that contains all these keys.

In [ ]:
# ADDED CELL: reproducibility seed
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Seed:", SEED)

In [ ]:
# CHANGED CELL: processor/model keep same architecture, now with your label mapping
processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

We'll first create `id2label` and label2id mappings, useful for inference. Note that `LayoutLMv3ForTokenClassification` (the model we'll use later on) will simply output an integer index for a particular class (for each token), so we still need to map it to an actual class name.

In [ ]:
# CHANGED CELL: label mappings summary
print("NER label2id:", label2id)
print("doc_status label2id:", status2id)
print("num NER labels:", num_labels)
print("num doc-status labels:", ndoc_labels)

In [ ]:
print(label_list)

In [ ]:
print(id2label)

Next, we'll define a function which we can apply on the entire dataset.

In [ ]:
# CHANGED CELL: preprocessing for NER Trainer

def prepare_examples(examples):
    images = [img.convert("RGB") for img in examples["image"]]
    encoding = processor(
        images=images,
        text=examples["tokens"],
        boxes=examples["bboxes"],
        word_labels=examples["ner_tags"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    cleaned = {}
    for key, value in encoding.items():
        cleaned[key] = value.tolist() if hasattr(value, "tolist") else value
    return cleaned

In [ ]:
# ADDED CELL: torch/gpu sanity check for current kernel
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# CHANGED CELL: split sanity summary
for split_name in ["train", "validation", "test"]:
    print(f"{split_name} size:", len(dataset[split_name]))
    print(f"{split_name} doc_status:", dict(Counter(dataset[split_name]["doc_status"])))

In [ ]:
encoded_dataset = dataset.map(
    prepare_examples,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Encoding dataset",
)

encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "bbox", "labels", "pixel_values"],
)

train_dataset = encoded_dataset["train"]
eval_dataset = encoded_dataset["validation"]
test_dataset = encoded_dataset["test"]


In [ ]:
example = train_dataset[0]
processor.tokenizer.decode(example["input_ids"])

Next, we set the format to PyTorch.

In [ ]:
# CHANGED CELL: dataset is already in torch format from encoded_dataset.set_format(...)
print("train_dataset format:", train_dataset.format)

Let's verify that everything was created properly:

In [ ]:
import torch

example = train_dataset[0]
for k,v in example.items():
    print(k,v.shape)

In [ ]:
eval_dataset

In [ ]:
processor.tokenizer.decode(eval_dataset[0]["input_ids"])

In [ ]:
for id, label in zip(train_dataset[0]["input_ids"], train_dataset[0]["labels"]):
  print(processor.tokenizer.decode([id]), label.item())

## Define metrics

Next, we define a `compute_metrics` function, which is used by the Trainer to ... compute metrics.

This function should take a named tuple as input, and return a dictionary as output as stated in the [docs](https://huggingface.co/docs/transformers/main_classes/trainer).

In [ ]:
# CHANGED CELL: seqeval compute_metrics using evaluate (compatible with current versions)
seqeval = evaluate.load("seqeval")

In [ ]:
# CHANGED CELL: seqeval metrics for NER (overall + per-entity)
TARGET_ENTITIES = sorted(
    {
        label.split("-", 1)[1]
        for label in label_list
        if label != "O" and "-" in label
    }
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        pred_labels = []
        ref_labels = []
        for p, l in zip(pred, lab):
            if l != -100:
                pred_labels.append(id2label[int(p)])
                ref_labels.append(id2label[int(l)])
        true_predictions.append(pred_labels)
        true_labels.append(ref_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    metrics = {
        "precision": float(results.get("overall_precision", 0.0)),
        "recall": float(results.get("overall_recall", 0.0)),
        "f1": float(results.get("overall_f1", 0.0)),
        "accuracy": float(results.get("overall_accuracy", 0.0)),
    }

    for entity in TARGET_ENTITIES:
        stats = results.get(entity, results.get(entity.lower(), {}))
        metrics[f"{entity.lower()}_precision"] = float(stats.get("precision", 0.0))
        metrics[f"{entity.lower()}_recall"] = float(stats.get("recall", 0.0))
        metrics[f"{entity.lower()}_f1"] = float(stats.get("f1", 0.0))

    return metrics

## Define the model

Next we define the model: this is a Transformer encoder with pre-trained weights, and a randomly initialized head on top for token classification.

In [ ]:
# CHANGED CELL: processor/model keep same architecture, now with your label mapping
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

## Define TrainingArguments + Trainer

Next we define the `TrainingArguments`, which define all hyperparameters related to training. Note that there is a huge amount of parameters to tweak, check the [docs](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments) for more info.

In [ ]:
# CHANGED CELL: TrainingArguments for NER (early-stopping friendly + version-safe)
params = set(inspect.signature(TrainingArguments.__init__).parameters)

args = {
    "output_dir": "layoutlmv3_cnopt_v4_ner_output",
    "overwrite_output_dir": True,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 12,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "save_total_limit": 2,
    "logging_steps": 20,
    "seed": SEED,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "remove_unused_columns": False,
}

if "evaluation_strategy" in params:
    args["evaluation_strategy"] = "epoch"
elif "eval_strategy" in params:
    args["eval_strategy"] = "epoch"
elif "evaluate_during_training" in params:
    args["evaluate_during_training"] = True

if "save_strategy" in params:
    args["save_strategy"] = "epoch"
elif "save_steps" in params and "eval_steps" in params:
    args["save_steps"] = 50
    args["eval_steps"] = 50

if "lr_scheduler_type" in params:
    args["lr_scheduler_type"] = "cosine"

if "fp16" in params:
    args["fp16"] = torch.cuda.is_available()

if "report_to" in params:
    args["report_to"] = "none"

supported_args = {k: v for k, v in args.items() if k in params}

has_eval = (
    (supported_args.get("evaluation_strategy", "no") != "no")
    or (supported_args.get("eval_strategy", "no") != "no")
    or bool(supported_args.get("evaluate_during_training", False))
)
if not has_eval:
    supported_args.pop("load_best_model_at_end", None)
    supported_args.pop("metric_for_best_model", None)
    supported_args.pop("greater_is_better", None)

training_args = TrainingArguments(**supported_args)
print(training_args)

We can now instantiate a Trainer, with the model and args defined above. We also provide our datasets, as well as a "default data collator" - which will batch the examples using `torch.stack`. We also provide our `compute_metrics` function defined above.

In [ ]:
# CHANGED CELL: Trainer for NER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## Train the model

Let's train!

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
for name in ["dataset","processor","train_dataset","eval_dataset","model","training_args","trainer"]:
    print(name, "->", name in globals())


In [ ]:
# CHANGED CELL: train from scratch (no checkpoint resume)
trainer.train(resume_from_checkpoint=False)

## Evaluate the model

NOTE: we end up with an F1 score of about 90%. Here's what I got on a typical run:
```
Step	Training Loss	Val Loss	Precision	Recall	F1	Accuracy
100	  No log	  0.716025	0.752040	0.824143	0.786442	0.780459
200	  No log	  0.584986	0.828558	0.876304	0.851762	0.801973
300	  No log	  0.525926	0.859583	0.900149	0.879398	0.833947
400	  No log	  0.492821	0.881413	0.904620	0.892866	0.854630
500	  0.561200	0.528126	0.858382	0.885246	0.871607	0.852490
600	  0.561200	0.547107	0.888023	0.906110	0.896976	0.847973
700	  0.561200	0.555438	0.887338	0.915549	0.901222	0.859384
800	  0.561200	0.582942	0.881471	0.905117	0.893137	0.854749
900	  0.561200	0.599762	0.891051	0.910084	0.900467	0.852015
1000	0.133400	0.608207	0.887222	0.910581	0.898750	0.847855
````

However, this score cannot be directly compared to LayoutLM and LayoutLMv2, as LayoutLMv3 employs so-called **segment position embeddings** (inspired by [StructuralLM](https://arxiv.org/abs/2105.11210)). This means that several tokens that belong to the same "segment" (let's say, an address) get the same bounding box coordinates, and in return the same 2D position embeddings.

This is also mentioned in the paper:
>  Note that LayoutLMv3 and StructuralLM use segment-level layout positions, while the other works use word-level layout positions. The use of segment-level positions may benefit the semantic entity labeling task on FUNSD [25], so the two types of work are not directly comparable.

In [ ]:
# CHANGED CELL: evaluate NER on validation + test
val_metrics = trainer.evaluate()
test_metrics_ner = trainer.evaluate(test_dataset, metric_key_prefix="test")
print("NER validation metrics:", val_metrics)
print("NER test metrics:", test_metrics_ner)

## Inference

You can load the model for inference as follows:

In [ ]:
# CHANGED CELL: use freshly trained in-memory model for inference
model = trainer.model
model.eval()

Let's take an example of the training dataset to show inference.

In [ ]:
# CHANGED CELL: pick any split/sample for debug inference visualization
debug_split = "validation"  # "validation" or "test"
debug_index = 0

example = dataset[debug_split][debug_index]
print("split:", debug_split, "index:", debug_index)
print("id:", example["id"], "doc_status:", example.get("doc_status"))
print(example.keys())

We first prepare it for the model using the processor.

In [ ]:
image = example["image"]
words = example["tokens"]
boxes = example["bboxes"]

encoding = processor(image, words, boxes=boxes, return_tensors="pt", truncation=True, padding="max_length", max_length=512)
for k, v in encoding.items():
    print(k, v.shape)

Next, we do a forward pass. We use torch.no_grad() as we don't require gradient computation.

In [ ]:
with torch.no_grad():
  outputs = model(**encoding)

The model outputs logits of shape (batch_size, seq_len, num_labels).

In [ ]:
import torch

logits = outputs.logits
probs = torch.softmax(logits, dim=-1).squeeze(0)
logits.shape


We take the highest score for each token, using argmax. This serves as the predicted label for each token.

In [ ]:
pred_ids = probs.argmax(-1).tolist()
conf_scores = probs.max(dim=-1).values.tolist()

# Map token predictions back to word predictions (use first sub-token per word)
word_ids = encoding.word_ids(batch_index=0)
word_pred_labels = ["O"] * len(words)
word_pred_scores = [0.0] * len(words)
seen_word = set()

for token_idx, word_idx in enumerate(word_ids):
    if word_idx is None or word_idx in seen_word:
        continue
    seen_word.add(word_idx)
    word_pred_labels[word_idx] = id2label[pred_ids[token_idx]]
    word_pred_scores[word_idx] = conf_scores[token_idx]

print("token preds:", len(pred_ids), "word preds:", len(word_pred_labels))

Let's compare this to the ground truth: note that many labels are -100, as we're only labeling the first subword token of each word.

NOTE: at "true inference" time, you don't have access to labels, see the latest section of this notebook how you can use `offset_mapping` in that case.

In [ ]:
print("No ground-truth labels in inference encoding.")


So let's only compare predictions and labels at positions where the label isn't -100. We also want to have the bounding boxes of these (unnormalized):

In [ ]:
def unnormalize_box(bbox, width, height):
    return [
        width * (bbox[0] / 1000),
        height * (bbox[1] / 1000),
        width * (bbox[2] / 1000),
        height * (bbox[3] / 1000),
    ]

MIN_CONFIDENCE = 0.35  # debug threshold
width, height = image.size

true_predictions = []
true_boxes = []
true_scores = []

for label, score, box in zip(word_pred_labels, word_pred_scores, boxes):
    if label == "O" or score < MIN_CONFIDENCE:
        continue
    true_predictions.append(label)
    true_boxes.append(unnormalize_box(box, width, height))
    true_scores.append(score)

In [ ]:
from PIL import ImageDraw, ImageFont

draw = ImageDraw.Draw(image)
font = ImageFont.load_default()

def iob_to_label(label):
    if label == "O":
        return "other"
    return label.split("-", 1)[-1].lower() if "-" in label else str(label).lower()

if "label2color" not in globals():
    palette = ["blue", "green", "orange", "purple", "red", "brown", "teal", "magenta", "black"]
    entity_tags = sorted({iob_to_label(lbl) for lbl in label_list})
    label2color = {tag: palette[i % len(palette)] for i, tag in enumerate(entity_tags)}
    label2color.setdefault("other", "gray")


In [ ]:
# ADDED CELL: prediction visualization (boxes + labels + confidence)
for prediction, box, score in zip(true_predictions, true_boxes, true_scores):
    predicted_label = iob_to_label(prediction)
    if predicted_label == "other":
        continue

    color = label2color.get(predicted_label, "gray")
    draw.rectangle(box, outline=color, width=2)
    draw.text(
        (box[0] + 4, max(0, box[1] - 12)),
        text=f"{predicted_label}:{score:.2f}",
        fill=color,
        font=font,
    )

image

Compare this to the ground truth:

In [ ]:
image = example["image"].convert("RGB")
draw = ImageDraw.Draw(image)
width, height = image.size

for word, box, label in zip(example["tokens"], example["bboxes"], example["ner_tags"]):
    actual_label = iob_to_label(id2label[label])
    box = unnormalize_box(box, width, height)
    color = label2color.get(actual_label, "gray")
    draw.rectangle(box, outline=color, width=2)
    draw.text((box[0] + 10, box[1] - 10), actual_label, fill=color, font=font)

image

## Note: inference when you don't have labels

The code above used the `labels` to determine which tokens were at the start of a particular word or not. Of course, at inference time, you don't have access to any labels. In that case, you can leverage the `offset_mapping` returned by the tokenizer. I do have a notebook for that (for LayoutLMv2, but it's equivalent for LayoutLMv3) [here](https://github.com/NielsRogge/Transformers-Tutorials/blob/master/LayoutLMv2/FUNSD/True_inference_with_LayoutLMv2ForTokenClassification_%2B_Gradio_demo.ipynb).

In [ ]:
# ADDED CELL: train a document-status classifier (valid/suspicious/incomplete/wrong_type)
import inspect


def prepare_doc_examples(examples):
    images = [img.convert("RGB") for img in examples["image"]]
    encoding = processor(
        images=images,
        text=examples["tokens"],
        boxes=examples["bboxes"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    encoding["labels"] = examples["doc_status_id"]

    cleaned = {}
    for key, value in encoding.items():
        cleaned[key] = value.tolist() if hasattr(value, "tolist") else value
    return cleaned


doc_encoded_dataset = dataset.map(
    prepare_doc_examples,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Encoding doc-status dataset",
)

doc_encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "bbox", "pixel_values", "labels"],
)

doc_train_dataset = doc_encoded_dataset["train"]
doc_eval_dataset = doc_encoded_dataset["validation"]
doc_test_dataset = doc_encoded_dataset["test"]

DOC_STATUS_TARGETS = [id2status[i] for i in range(len(id2status))]


def _safe_div(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else 0.0


def compute_doc_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    metrics = {
        "accuracy": float((preds == labels).mean()),
    }

    per_class_f1 = []
    for class_id, class_name in enumerate(DOC_STATUS_TARGETS):
        tp = int(((preds == class_id) & (labels == class_id)).sum())
        fp = int(((preds == class_id) & (labels != class_id)).sum())
        fn = int(((preds != class_id) & (labels == class_id)).sum())

        precision = _safe_div(tp, tp + fp)
        recall = _safe_div(tp, tp + fn)
        f1 = _safe_div(2 * precision * recall, precision + recall)

        metrics[f"{class_name}_precision"] = precision
        metrics[f"{class_name}_recall"] = recall
        metrics[f"{class_name}_f1"] = f1
        per_class_f1.append(f1)

    metrics["macro_f1"] = float(np.mean(per_class_f1)) if per_class_f1 else 0.0
    return metrics


doc_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(doc_status_list),
    id2label=doc_id2label,
    label2id=doc_label2id,
)

doc_params = set(inspect.signature(TrainingArguments.__init__).parameters)
doc_args = {
    "output_dir": "layoutlmv3_cnopt_v4_doc_status_output",
    "overwrite_output_dir": True,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 10,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "save_total_limit": 2,
    "logging_steps": 20,
    "seed": SEED,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "remove_unused_columns": False,
}

if "evaluation_strategy" in doc_params:
    doc_args["evaluation_strategy"] = "epoch"
elif "eval_strategy" in doc_params:
    doc_args["eval_strategy"] = "epoch"
elif "evaluate_during_training" in doc_params:
    doc_args["evaluate_during_training"] = True

if "save_strategy" in doc_params:
    doc_args["save_strategy"] = "epoch"
elif "save_steps" in doc_params and "eval_steps" in doc_params:
    doc_args["save_steps"] = 50
    doc_args["eval_steps"] = 50

if "fp16" in doc_params:
    doc_args["fp16"] = torch.cuda.is_available()

if "report_to" in doc_params:
    doc_args["report_to"] = "none"

doc_supported_args = {k: v for k, v in doc_args.items() if k in doc_params}

doc_has_eval = (
    (doc_supported_args.get("evaluation_strategy", "no") != "no")
    or (doc_supported_args.get("eval_strategy", "no") != "no")
    or bool(doc_supported_args.get("evaluate_during_training", False))
)
if not doc_has_eval:
    doc_supported_args.pop("load_best_model_at_end", None)
    doc_supported_args.pop("metric_for_best_model", None)
    doc_supported_args.pop("greater_is_better", None)

doc_training_args = TrainingArguments(**doc_supported_args)

doc_trainer = Trainer(
    model=doc_model,
    args=doc_training_args,
    train_dataset=doc_train_dataset,
    eval_dataset=doc_eval_dataset,
    data_collator=default_data_collator,
    compute_metrics=compute_doc_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

doc_trainer.train(resume_from_checkpoint=False)
doc_val_metrics = doc_trainer.evaluate()
doc_test_metrics = doc_trainer.evaluate(doc_test_dataset, metric_key_prefix="test")

print("Doc-status validation metrics:", doc_val_metrics)
print("Doc-status test metrics:", doc_test_metrics)

In [ ]:
# CHANGED CELL: final CNOPT gate + fraud decision fusion (NER + doc-status model)
import json

REQUIRED_ENTITIES = ["AUTHORITY", "NAME", "REG_NUMBER", "ISSUE_DATE", "COUNTRY", "TITLE", "SIGNATURE", "STAMP"]
STRICT_ENTITIES_FOR_ACCEPT = ["AUTHORITY", "NAME", "REG_NUMBER", "ISSUE_DATE", "COUNTRY", "TITLE"]


def _label_to_entity(tag):
    if tag == "O":
        return None
    return tag.split("-", 1)[1] if "-" in tag else tag


def _merge_entity_chunks(words, boxes, labels, scores):
    chunks = {}
    active = {}

    for word, box, tag, score in zip(words, boxes, labels, scores):
        entity = _label_to_entity(tag)
        if entity is None:
            continue

        prefix = tag.split("-", 1)[0] if "-" in tag else "B"
        if prefix == "B" or entity not in active:
            chunk = {"text": [word], "boxes": [box], "scores": [score]}
            chunks.setdefault(entity, []).append(chunk)
            active[entity] = chunk
        else:
            active[entity]["text"].append(word)
            active[entity]["boxes"].append(box)
            active[entity]["scores"].append(score)

    merged = {}
    for entity, items in chunks.items():
        merged[entity] = []
        for item in items:
            xs0 = [b[0] for b in item["boxes"]]
            ys0 = [b[1] for b in item["boxes"]]
            xs1 = [b[2] for b in item["boxes"]]
            ys1 = [b[3] for b in item["boxes"]]
            merged[entity].append(
                {
                    "text": " ".join(item["text"]).strip(),
                    "bbox": [min(xs0), min(ys0), max(xs1), max(ys1)],
                    "score": float(np.mean(item["scores"])),
                }
            )
    return merged


def predict_ner_entities(example_row):
    image = example_row["image"].convert("RGB")
    words = example_row["tokens"]
    boxes = example_row["bboxes"]

    encoded = processor(
        image,
        words,
        boxes=boxes,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    word_ids = encoded.word_ids(batch_index=0)

    device = next(model.parameters()).device
    model_inputs = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = model(**model_inputs).logits

    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu()
    pred_ids = probs.argmax(-1).tolist()
    conf_scores = probs.max(dim=-1).values.tolist()

    word_labels = ["O"] * len(words)
    word_scores = [0.0] * len(words)
    seen_word = set()

    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx in seen_word:
            continue
        seen_word.add(word_idx)
        word_labels[word_idx] = id2label[int(pred_ids[token_idx])]
        word_scores[word_idx] = float(conf_scores[token_idx])

    entities = _merge_entity_chunks(words, boxes, word_labels, word_scores)
    return entities, word_labels, word_scores


def predict_doc_status(example_row):
    image = example_row["image"].convert("RGB")
    words = example_row["tokens"]
    boxes = example_row["bboxes"]

    encoded = processor(
        image,
        words,
        boxes=boxes,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    device = next(doc_model.parameters()).device
    model_inputs = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = doc_model(**model_inputs).logits

    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    pred_idx = int(np.argmax(probs))
    pred_status = id2status[pred_idx]
    pred_conf = float(probs[pred_idx])

    full_probs = {id2status[i]: float(probs[i]) for i in range(len(probs))}
    return pred_status, pred_conf, full_probs


def _entity_text(entities, key):
    return " ".join([item.get("text", "") for item in entities.get(key, [])]).strip()


def _regex_any(text, patterns):
    return any(re.search(pattern, text or "", flags=re.IGNORECASE) for pattern in patterns)


def build_fraud_flags(entities):
    presence = {entity: bool(entities.get(entity)) for entity in REQUIRED_ENTITIES}
    missing = [entity for entity, ok in presence.items() if not ok]
    flags = []

    title_authority_text = (_entity_text(entities, "TITLE") + " " + _entity_text(entities, "AUTHORITY")).strip()
    cnopt_patterns = [
        r"\bcnop\b",
        r"\bcnopt\b",
        r"\bordre\b",
        r"\bpharmac",
        r"\bcertificat\b",
        r"\binscription\b",
    ]
    if not _regex_any(title_authority_text, cnopt_patterns):
        flags.append("cnopt_keywords_missing")

    reg_text = _entity_text(entities, "REG_NUMBER")
    if not reg_text:
        flags.append("reg_number_missing")
    elif not _regex_any(reg_text, [r"\d{3,}", r"[A-Za-z]{1,5}[- ]?\d{2,}"]):
        flags.append("reg_number_format_unusual")

    if not presence["SIGNATURE"] and not presence["STAMP"]:
        flags.append("signature_and_stamp_missing")

    severe_flags = [flag for flag in flags if flag in {"cnopt_keywords_missing"}]
    return presence, missing, flags, severe_flags


def fuse_decision(doc_pred_status, doc_conf, presence, missing, flags, severe_flags):
    base_risk = {
        "valid": 0.15,
        "suspicious": 0.75,
        "incomplete": 0.65,
        "wrong_type": 0.95,
    }.get(doc_pred_status, 0.60)

    risk = min(1.0, base_risk + 0.08 * len(flags) + 0.10 * len(severe_flags))

    strict_missing = [k for k in STRICT_ENTITIES_FOR_ACCEPT if not presence.get(k, False)]
    wrong_type_trigger = (doc_pred_status == "wrong_type" and doc_conf >= 0.50) or ("cnopt_keywords_missing" in severe_flags)

    if wrong_type_trigger:
        decision = "REJECT"
        final_status = "wrong_type"
    elif doc_pred_status in ("suspicious", "incomplete"):
        decision = "MANUAL_REVIEW"
        final_status = doc_pred_status
    elif strict_missing:
        decision = "MANUAL_REVIEW"
        final_status = "incomplete"
    elif doc_conf < 0.55:
        decision = "MANUAL_REVIEW"
        final_status = "suspicious"
    else:
        decision = "ACCEPT"
        final_status = "valid"

    return decision, final_status, float(risk)


inference_split = "test"
inference_index = 0
example = dataset[inference_split][inference_index]

entities, word_pred_labels, word_pred_scores = predict_ner_entities(example)
doc_pred_status, doc_conf, doc_probs = predict_doc_status(example)
presence, missing, flags, severe_flags = build_fraud_flags(entities)
decision, final_status, fraud_risk_score = fuse_decision(
    doc_pred_status,
    doc_conf,
    presence,
    missing,
    flags,
    severe_flags,
)

payload = {
    "id": example["id"],
    "ground_truth_doc_status": example.get("doc_status", "unknown"),
    "document_decision": decision,
    "fraud_status": final_status,
    "fraud_risk_score": round(fraud_risk_score, 4),
    "is_cnopt_document": final_status != "wrong_type",
    "doc_status_model_prediction": doc_pred_status,
    "doc_status_model_confidence": round(doc_conf, 4),
    "doc_status_model_probs": {k: round(v, 4) for k, v in doc_probs.items()},
    "missing_entities": missing,
    "flags": flags,
    "entity_presence": presence,
    "entities": entities,
}

print(json.dumps(payload, ensure_ascii=False, indent=2))

In [ ]:
# ADDED CELL: evaluate final fraud pipeline on full test split

def _macro_f1(y_true, y_pred, labels):
    f1s = []
    for label in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1s.append(f1)
    return float(np.mean(f1s)) if f1s else 0.0


def evaluate_fraud_pipeline(split_name="test", max_docs=None):
    y_true = []
    y_pred = []
    decision_counts = Counter()

    split_ds = dataset[split_name]
    n = len(split_ds) if max_docs is None else min(len(split_ds), max_docs)

    for idx in range(n):
        row = split_ds[idx]
        entities, _, _ = predict_ner_entities(row)
        doc_pred_status, doc_conf, _ = predict_doc_status(row)
        presence, missing, flags, severe_flags = build_fraud_flags(entities)
        decision, final_status, _ = fuse_decision(doc_pred_status, doc_conf, presence, missing, flags, severe_flags)

        y_true.append(row.get("doc_status", "valid"))
        y_pred.append(final_status)
        decision_counts[decision] += 1

    accuracy = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)
    macro_f1 = _macro_f1(y_true, y_pred, doc_status_list)

    confusion = {label: {l2: 0 for l2 in doc_status_list} for label in doc_status_list}
    for t, p in zip(y_true, y_pred):
        if t in confusion and p in confusion[t]:
            confusion[t][p] += 1

    return {
        "samples": len(y_true),
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "decision_counts": dict(decision_counts),
        "confusion": confusion,
    }


pipeline_test_metrics = evaluate_fraud_pipeline("test")
print(json.dumps(pipeline_test_metrics, indent=2))